In [1]:
# ═══════════════════════════════════════
# TRY 2 — STANDARD SESSION OPENER
# ═══════════════════════════════════════

import json, os, pickle, random, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = "/kaggle/input/datasets/aadityaupadhyay1353"

CFG_PATH = f"{BASE}/cps-config/config.json"

with open(CFG_PATH) as f:
    CFG = json.load(f)

TRAIN_IDS = CFG["engine_partition"]["train"]
VAL_IDS = CFG["engine_partition"]["validation"]
TEST_IDS = CFG["engine_partition"]["test"]

OUT = "/kaggle/working/"
os.makedirs(OUT, exist_ok=True)

print("TRY 2 session ready")
print("Train engines:", len(TRAIN_IDS))
print("Val engines:", len(VAL_IDS))
print("Test engines:", len(TEST_IDS))

TRY 2 session ready
Train engines: 70
Val engines: 15
Test engines: 15


In [2]:
df = pd.read_csv(
    f"{BASE}/v2-phase2-labeled/v2_phase2_labeled_dataset.csv"
)

print("Dataset shape:", df.shape)

sensor_cols = [c for c in df.columns if c.startswith("s")]

print("Sensor columns:", sensor_cols)
print("Total sensors:", len(sensor_cols))

Dataset shape: (20631, 23)
Sensor columns: ['s2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
Total sensors: 14


In [3]:
ROLL_W = 5

for s in sensor_cols:

    grp = df.groupby("engine_id")[s]

    df[f"{s}_mean5"] = grp.transform(
        lambda x: x.rolling(ROLL_W, min_periods=1).mean()
    )

    df[f"{s}_std5"] = grp.transform(
        lambda x: x.rolling(ROLL_W, min_periods=1).std()
    )

    df[f"{s}_slope5"] = grp.transform(
        lambda x: x.rolling(ROLL_W, min_periods=1).apply(
            lambda v: np.polyfit(range(len(v)), v, 1)[0] if len(v) > 1 else 0,
            raw=True,
        )
    )

print("Shape after features:", df.shape)

Shape after features: (20631, 65)


In [4]:
print("NaN count before fill:", df.isnull().sum().sum())

df.fillna(0, inplace=True)

print("NaN count after fill:", df.isnull().sum().sum())

NaN count before fill: 1400
NaN count after fill: 0


In [5]:
EXCLUDE = [
    "engine_id",
    "cycle",
    "max_cycle",
    "op1",
    "op2",
    "op3",
    "label_w10",
    "label_w20",
    "label_w30",
]

feature_cols = [c for c in df.columns if c not in EXCLUDE]

print("Total features:", len(feature_cols))

assert "max_cycle" not in feature_cols, "LEAKAGE: max_cycle detected!"

print("ASSERTION PASSED — max_cycle NOT in feature set")

Total features: 56
ASSERTION PASSED — max_cycle NOT in feature set


In [6]:
from sklearn.preprocessing import MinMaxScaler

train_df = df[df["engine_id"].isin(TRAIN_IDS)]

scaler = MinMaxScaler()

scaler.fit(train_df[feature_cols])

print("Scaler fitted on training partition")

Scaler fitted on training partition


In [7]:
import json

with open(f"{OUT}v2_feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

print("Saved v2_feature_cols.json")

Saved v2_feature_cols.json


In [8]:
import pickle

pickle.dump(scaler, open(f"{OUT}v2_scaler.pkl", "wb"))

print("Saved v2_scaler.pkl")

Saved v2_scaler.pkl


In [9]:
df.to_csv(f"{OUT}v2_phase3_feature_dataset.csv", index=False)

print("Saved v2_phase3_feature_dataset.csv")

Saved v2_phase3_feature_dataset.csv


In [10]:
!ls -lh /kaggle/working/

total 17M
---------- 1 root root  13K Apr 12 12:05 __notebook__.ipynb
-rw-r--r-- 1 root root  620 Apr 12 12:05 v2_feature_cols.json
-rw-r--r-- 1 root root  17M Apr 12 12:05 v2_phase3_feature_dataset.csv
-rw-r--r-- 1 root root 3.4K Apr 12 12:05 v2_scaler.pkl
